In [ ]:
# IMPORTS
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import geopandas as gpd
from shapely.geometry import Point
from sklearn.cluster import KMeans
from sklearn.preprocessing import MinMaxScaler

In [ ]:
# CHARGEMENT DES DONNÉES
bornes = pd.read_csv("../data/raw/BornesPropres.csv", low_memory=False) # Données bornes
vehicules = pd.read_csv("../data/raw/voitures_par_commune_cleaned.csv", low_memory=False) # Données parc de véhicules
population = pd.read_csv("../data/clean/population_with_geopoint.csv", low_memory=False) # Données population

In [ ]:
# CALIBRAGE DES DONNEES

# Fiabiliser les colonnes BORNES
bornes["date_mise_en_service"] = pd.to_datetime(bornes["date_mise_en_service"], errors="coerce") # date

# Fiabiliser les colonnes VEHICULES
vehicules["code_commune"] = vehicules["code_commune"].astype(str).str.zfill(5)  # code commune sur 5 chiffres
vehicules["date_recensement"] = pd.to_datetime(vehicules["date_recensement"], errors="coerce") # date
vehicules["nb_voit_elec"] = pd.to_numeric(vehicules["nb_voit_elec"], errors="coerce") # nb voitures elec
vehicules["nb_voit_total"] = pd.to_numeric(vehicules["nb_voit_total"], errors="coerce") # nb voitures

# Fiabiliser les colonnes POPULATION
cols_to_num = [
    "0 à 19 ans", "20 à 39 ans", "40 à 59 ans",
    "60 à 74 ans", "75 ans et plus", "Total"
]

for col in cols_to_num:
    population[col] = (
        population[col]
        .astype(str)                              # force en string pour nettoyage
        .str.replace(r"\s", "", regex=True)       # supprime tous les espaces
        .str.replace(",", ".", regex=False)       # remplace les virgules par des points si besoin
        .replace({"-": None, "nc": None})         # gère les valeurs non numériques
        .astype(float)                            # convertit en float
    )

In [ ]:
# JOINTURE VEHICULES & POPULATION
    # Harmonisation des clés de jointure # 1
    # Clef : vehicules = departement / population = Département

# Harmoniser les noms de colonnes département
population_df = population.rename(columns={'Département': 'departement'})
vehicules_df = vehicules.rename(columns={'departement': 'departement'})

# Standardiser les valeurs département (str et majuscules)
for df in [population_df, vehicules_df]:
    df['departement'] = df['departement'].astype(str).str.strip().str.upper()

# Vérifier les départements communs
set_pop = set(population_df['departement'].unique())
set_vehicules = set(vehicules_df['departement'].unique())

print("Départements communs population / véhicules :", len(set_pop & set_vehicules))

In [ ]:
    # MERGE :
merged_vehicules_pop = vehicules_df.merge(population_df, on='departement', how='inner')

# Vérification
print(merged_vehicules_pop.head())

In [ ]:
# JOINTURE BORNES & POPULATION
    # Harmonisation des clés de jointure BORNES
    # Clef bornes = Geopoint

# Transformer la colonne Geopoint en 2 colonnes
coord_split = (
    bornes['Geopoint']
    .fillna('')                                 # évite NaN
    .str.strip()                                # enlève crochets et espaces
    .str.replace('POINT\(|\)', '', regex=True)  # supprime "POINT(" et ")"
    .str.split(' ', n=1, expand=True)           # split sur espace en 2 colonnes
)

# Renommer les 2 colonnes
coord_split.columns = ['lon', 'lat']

# Attribuer les données aux bonnes colonnes
bornes['lon'] = coord_split['lon']
bornes['lat'] = coord_split['lat']

# Conversion en float dans bornes
bornes['lon'] = pd.to_numeric(bornes['lon'], errors='coerce')
bornes['lat'] = pd.to_numeric(bornes['lat'], errors='coerce')

In [ ]:
# Harmonisation des clés de jointure POPULATION
    # Clef population = longitude_decimal + latitude_decimal

# Renommer les colonnes lon et lat
population_def = population.rename(columns={'latitude_decimal' : 'lat', 'longitude_decimal': 'lon'})

# Standardiser les valeurs lon, lat (float) de population
population_def[['lon', 'lat']] = (
    population_def[['lon', 'lat']]
    .astype(str)
    .replace(r"\s", "", regex=True)     # supprime espaces
    .replace(",", ".", regex=False)     # virgule → point
    .replace({"-": None, "nc": None})
    .astype(float)
)

In [ ]:
    # TRANSFORMATION GEO

# Créer des GeoDataFrames (bornes & populations)
gdf_bornes = gpd.GeoDataFrame(
    bornes,
    geometry=gpd.points_from_xy(bornes['lon'], bornes['lat']),
    crs="EPSG:4326"
)

gdf_population = gpd.GeoDataFrame(
    population_def,
    geometry=gpd.points_from_xy(population_def['lon'], population_def['lat']),
    crs="EPSG:4326"
)

# Reprojection dans un CRS métrique adapté (Lambert 93 pour la France)
gdf_bornes_proj = gdf_bornes.to_crs("EPSG:2154")
gdf_population_proj = gdf_population.to_crs("EPSG:2154")


In [ ]:
# MERGE 
# ❌ pop vide

# Jointure par plus proche voisin avec une distance max (ex: 5000 mètres)
gdf_merged_proj = gpd.sjoin_nearest(
    gdf_bornes_proj,
    gdf_population_proj,
    how='left',
    max_distance=5000  # distance max en mètres
)

# Reprojection en EPSG:4326 si tu veux continuer avec lat/lon classiques
gdf_merged = gdf_merged_proj.to_crs("EPSG:4326")

In [ ]:
# VERIFICATION :
print(gdf_merged.head(5))

# Créer set_bornes en utilisant les départements uniques du DataFrame fusionné
set_bornes = set(gdf_merged['Département'].dropna().unique())

# Vérification des départements communs (utiliser population_def pour avoir 'Département')
set_pop_original = set(population_def['Département'].unique())

print("Départements communs population / bornes :", len(set_pop_original & set_bornes))

In [ ]:
# JOINTURE BORNES & POPULATION PAR DÉPARTEMENT

# Harmonisation des noms de colonnes pour la jointure par département
# Note: gdf_merged contient déjà les bornes avec coordonnées population
# Ici on fait une analyse agrégée par département

# Renommer les colonnes pour homogénéiser
gdf_merged_renamed = gdf_merged.rename(columns={"Département": "departement"})

# Standardisation des valeurs département
gdf_merged_renamed["departement"] = gdf_merged_renamed["departement"].astype(str).str.strip().str.upper()
population_def["departement"] = population_def["Département"].astype(str).str.strip().str.upper()

# Vérification des départements
print("Départements dans bornes :", sorted(gdf_merged_renamed["departement"].dropna().unique())[:5])
print("Départements dans population :", sorted(population_def["departement"].unique())[:5])

# Agrégations par département
bornes_par_dep = gdf_merged_renamed.groupby("departement").size().reset_index(name="nb_bornes")
pop_par_dep = population_def.groupby("departement")["Total"].sum().reset_index(name="population_totale")

# Jointure des agrégations
df_dep = bornes_par_dep.merge(pop_par_dep, on="departement", how="inner")

# Calcul densité bornes / population
df_dep["bornes_pour_1000_hab"] = (df_dep["nb_bornes"] / df_dep["population_totale"]) * 1000

print(f"Données départementales créées : {len(df_dep)} départements")
print("Échantillon densité bornes :")
print(df_dep.head(3))

ÉTAT DES LIEUX – NIVEAU ACTUEL DE DÉPLOIEMENT 

In [ ]:
# ÉTAT DES LIEUX – NIVEAU ACTUEL DE DÉPLOIEMENT

print("\n" + "="*50)
print("📊 KPI GÉNÉRAUX")
print("="*50)

# Compléter la jointure véhicules-population d'abord
merged_vehicules_pop = vehicules_df.merge(population_df, on='departement', how='inner')
print(f"Jointure véhicules-population réussie : {len(merged_vehicules_pop)} lignes")

# 1. Nombre total de véhicules électriques
nb_ve = vehicules_df["nb_voit_elec"].sum()
print(f'🚗 Nombre total de VE : {nb_ve:,}')

# 2. Nombre total de bornes (utiliser une colonne unique identifiant)
# Vérifier les colonnes disponibles pour l'identifiant unique
if 'id_station_itinerance' in gdf_merged.columns:
    nb_bornes = gdf_merged["id_station_itinerance"].nunique()
elif 'id_station_local' in gdf_merged.columns:
    nb_bornes = gdf_merged["id_station_local"].nunique()
else:
    nb_bornes = len(gdf_merged)  # fallback
    
print(f'🔌 Nombre total de bornes : {nb_bornes:,}')

# 3. Ratio bornes / VE (comparaison avec recommandation AFI : 1 borne / 10 VE)
if nb_ve > 0:
    ratio_bornes_ve = nb_bornes / nb_ve
    ratio_pour_10_ve = ratio_bornes_ve * 10
    print(f'⚡ Ratio bornes / VE : {ratio_bornes_ve:.3f} (soit {ratio_pour_10_ve:.1f} bornes pour 10 VE)')
    
    # Comparaison avec la recommandation AFI
    if ratio_pour_10_ve >= 1:
        print("✅ Objectif AFI atteint (≥1 borne pour 10 VE)")
    else:
        deficit = 1 - ratio_pour_10_ve
        print(f"❌ Déficit par rapport à l'objectif AFI : {deficit:.1f} borne(s) manquante(s) pour 10 VE")

# 4. Densité moyenne bornes / population
densite_moyenne = df_dep["bornes_pour_1000_hab"].mean()
print(f'🏘️  Densité moyenne : {densite_moyenne:.2f} bornes pour 1000 habitants')

In [ ]:
# Top 5 et bottom 5 des départements
print(f'\n🏆 Top 5 départements (densité bornes/1000 hab) :')
top_5 = df_dep.nlargest(5, 'bornes_pour_1000_hab')[['departement', 'bornes_pour_1000_hab', 'nb_bornes']]
for _, row in top_5.iterrows():
    print(f"   {row['departement']} : {row['bornes_pour_1000_hab']:.2f} ({row['nb_bornes']} bornes)")

print(f'\n📉 Bottom 5 départements (densité bornes/1000 hab) :')
bottom_5 = df_dep.nsmallest(5, 'bornes_pour_1000_hab')[['departement', 'bornes_pour_1000_hab', 'nb_bornes']]
for _, row in bottom_5.iterrows():
    print(f"   {row['departement']} : {row['bornes_pour_1000_hab']:.2f} ({row['nb_bornes']} bornes)")

ANALYSE TEMPORELLE

In [ ]:
# ANALYSE TEMPORELLE

print("\n" + "="*50)
print("📈 ÉVOLUTION TEMPORELLE")
print("="*50)

# Évolution du parc de VE par date
vehicules_clean = vehicules_df.dropna(subset=['date_recensement', 'nb_voit_elec'])
evolution_ve = (vehicules_clean
                .groupby('date_recensement')['nb_voit_elec']
                .sum()
                .reset_index())

evolution_ve = evolution_ve.sort_values('date_recensement')
print(f"📊 Évolution VE sur {len(evolution_ve)} périodes")


# Évolution des bornes par date de mise en service
bornes_clean = gdf_merged.dropna(subset=['date_mise_en_service'])
evolution_bornes = (bornes_clean
                   .groupby(bornes_clean['date_mise_en_service'].dt.to_period('M'))
                   .size()
                   .reset_index())

evolution_bornes.columns = ['periode', 'nb_bornes_installees']
evolution_bornes['nb_bornes_cumulees'] = evolution_bornes['nb_bornes_installees'].cumsum()

print(f"🔌 Évolution bornes sur {len(evolution_bornes)} mois")


# Statistiques récentes
if len(evolution_ve) > 1:
    croissance_ve = ((evolution_ve['nb_voit_elec'].iloc[-1] - evolution_ve['nb_voit_elec'].iloc[0]) 
                     / evolution_ve['nb_voit_elec'].iloc[0] * 100)
    print(f"📈 Croissance VE totale : +{croissance_ve:.1f}%")

if len(evolution_bornes) > 12:
    bornes_12_derniers_mois = evolution_bornes['nb_bornes_installees'].tail(12).sum()
    print(f"🔌 Bornes installées (12 derniers mois) : {bornes_12_derniers_mois:,}")

print(f"\n✅ Analyse terminée - Données prêtes pour visualisation")

TOP 3 VISUALISATIONS - LES PLUS PERTINENTES 📈

In [ ]:
# TOP 3 VISUALISATIONS - LES PLUS PERTINENTES 📈

print("\n" + "="*50)
print("📊 TOP 3 VISUALISATIONS")
print("="*50)

# Configuration matplotlib
plt.style.use('default')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 11

# PRÉPARATION DONNÉES : DERNIER RECENSEMENT PAR ANNÉE 🟢

# 1. VE : Prendre le dernier recensement de chaque année
vehicules_clean = vehicules_df.dropna(subset=['date_recensement', 'nb_voit_elec']).copy()
vehicules_clean['annee'] = vehicules_clean['date_recensement'].dt.year

# Garder seulement le dernier recensement de chaque année
evolution_ve_annuelle = (vehicules_clean
                        .sort_values(['annee', 'date_recensement'])
                        .groupby('annee')
                        .last()  # Dernier recensement de l'année
                        .reset_index())

evolution_ve_annuelle = evolution_ve_annuelle.groupby('annee')['nb_voit_elec'].sum().reset_index()

print(f"✅ VE : {len(evolution_ve_annuelle)} années de données (dernier recensement)")

# 2. Bornes : Compter les installations cumulées par année
bornes_clean = gdf_merged.dropna(subset=['date_mise_en_service']).copy()
bornes_clean['annee_installation'] = bornes_clean['date_mise_en_service'].dt.year

# Cumul des bornes installées jusqu'à chaque année
bornes_par_annee = (bornes_clean
                   .groupby('annee_installation')
                   .size()
                   .reset_index(name='nouvelles_bornes'))

bornes_par_annee['bornes_cumulees'] = bornes_par_annee['nouvelles_bornes'].cumsum()

print(f"✅ Bornes : {len(bornes_par_annee)} années de données")

# 3. Données départementales pour analyse comparative
ve_par_dep = vehicules_df.groupby('departement')['nb_voit_elec'].sum().reset_index()
ve_par_dep['departement'] = ve_par_dep['departement'].astype(str).str.strip().str.upper()

df_complet = df_dep.merge(ve_par_dep, on='departement', how='inner')
df_complet['ratio_bornes_ve'] = df_complet['nb_bornes'] / df_complet['nb_voit_elec'].replace(0, np.nan)

GRAPHIQUE 1 : ÉVOLUTION VE vs BORNES 📈

In [ ]:
# ==========================================
# GRAPHIQUE 1 : ÉVOLUTION VE vs BORNES 📈
# ==========================================

fig, ax1 = plt.subplots(1, 1, figsize=(14, 8))

# Synchroniser les années communes
annees_communes = sorted(set(evolution_ve_annuelle['annee']) & set(bornes_par_annee['annee_installation']))

if len(annees_communes) >= 2:
    # Filtrer sur les années communes
    ve_sync = evolution_ve_annuelle[evolution_ve_annuelle['annee'].isin(annees_communes)]
    bornes_sync = bornes_par_annee[bornes_par_annee['annee_installation'].isin(annees_communes)]
    
    # Axe principal : VE
    color1 = 'tab:blue'
    ax1.set_xlabel('Année')
    ax1.set_ylabel('Nombre de Véhicules Électriques', color=color1)
    line1 = ax1.plot(ve_sync['annee'], ve_sync['nb_voit_elec'], 
                     marker='o', linewidth=3, markersize=8, color=color1, label='VE (dernier recensement)')
    ax1.tick_params(axis='y', labelcolor=color1)
    ax1.grid(True, alpha=0.3)
    
    # Axe secondaire : Bornes cumulées
    ax2 = ax1.twinx()
    color2 = 'tab:orange'
    ax2.set_ylabel('Nombre de Bornes (cumulé)', color=color2)
    line2 = ax2.plot(bornes_sync['annee_installation'], bornes_sync['bornes_cumulees'], 
                     marker='s', linewidth=3, markersize=8, color=color2, label='Bornes cumulées')
    ax2.tick_params(axis='y', labelcolor=color2)
    
    # Titre et légende
    ax1.set_title('Évolution du Parc VE vs Déploiement Bornes\n(Dernier recensement par année)', fontsize=14, pad=20)
    
    # Légende combinée
    lines = line1 + line2
    labels = [l.get_label() for l in lines]
    ax1.legend(lines, labels, loc='upper left')
    
    # Annotations des dernières valeurs
    if len(ve_sync) > 0:
        derniere_ve = ve_sync.iloc[-1]
        ax1.annotate(f'{int(derniere_ve["nb_voit_elec"]):,} VE', 
                    (derniere_ve['annee'], derniere_ve['nb_voit_elec']),
                    xytext=(10, 10), textcoords='offset points', 
                    bbox=dict(boxstyle='round,pad=0.3', facecolor=color1, alpha=0.3),
                    fontsize=10, ha='left')
    
    if len(bornes_sync) > 0:
        derniere_borne = bornes_sync.iloc[-1]
        ax2.annotate(f'{int(derniere_borne["bornes_cumulees"]):,} bornes', 
                    (derniere_borne['annee_installation'], derniere_borne['bornes_cumulees']),
                    xytext=(10, -20), textcoords='offset points', 
                    bbox=dict(boxstyle='round,pad=0.3', facecolor=color2, alpha=0.3),
                    fontsize=10, ha='left')

plt.tight_layout()
plt.show()

GRAPHIQUE 2 : TOP 15 DÉPARTEMENTS - PERFORMANCE 🏆

In [ ]:
# ==========================================
# GRAPHIQUE 2 : TOP 15 DÉPARTEMENTS - PERFORMANCE 🏆
# ==========================================

# Top 15 départements par nombre de VE
top_15 = df_complet.nlargest(15, 'nb_voit_elec').copy()

fig, ax = plt.subplots(1, 1, figsize=(16, 8))

# Créer le bar plot avec couleurs selon performance
colors = ['darkgreen' if ratio >= 0.1 else 'orange' if ratio >= 0.05 else 'red' 
          for ratio in top_15['ratio_bornes_ve'].fillna(0)]

bars = ax.bar(range(len(top_15)), top_15['nb_voit_elec'], 
              color=colors, alpha=0.7, edgecolor='black', linewidth=0.5)

# Ligne de référence pour l'objectif (à ajuster selon vos critères)
# ax.axhline(y=top_15['nb_voit_elec'].mean(), color='blue', linestyle='--', alpha=0.7, label='Moyenne nationale')

# Personnalisation
ax.set_xlabel('Départements')
ax.set_ylabel('Nombre de Véhicules Électriques')
ax.set_title('Top 15 Départements - Parc de Véhicules Électriques\n(Couleur selon ratio bornes/VE)', fontsize=14, pad=20)

# Labels des départements
ax.set_xticks(range(len(top_15)))
ax.set_xticklabels(top_15['departement'], rotation=45, ha='right')

# Ajouter les valeurs sur les barres + ratio
for i, (bar, row) in enumerate(zip(bars, top_15.itertuples())):
    # Nombre de VE
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + bar.get_height()*0.01, 
            f'{int(row.nb_voit_elec):,}', 
            ha='center', va='bottom', fontsize=9, weight='bold')
    
    # Ratio bornes/VE
    ratio_text = f'{row.ratio_bornes_ve:.3f}' if not pd.isna(row.ratio_bornes_ve) else 'N/A'
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height()*0.1, 
            f'R: {ratio_text}', 
            ha='center', va='bottom', fontsize=8, 
            bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.8))

# Légende des couleurs
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='darkgreen', label='Ratio ≥ 0.10 (Bon)'),
                   Patch(facecolor='orange', label='Ratio 0.05-0.10 (Moyen)'),
                   Patch(facecolor='red', label='Ratio < 0.05 (Faible)')]
ax.legend(handles=legend_elements, loc='upper right')

ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

GRAPHIQUE 3 : SCATTER - CORRÉLATION POPULATION vs VE vs BORNES 🎯

In [ ]:
# ==========================================
# GRAPHIQUE 3 : SCATTER - CORRÉLATION POPULATION vs VE vs BORNES 🎯
# ==========================================

fig, ax = plt.subplots(1, 1, figsize=(12, 8))

# Scatter plot avec taille des points = nombre de bornes
scatter = ax.scatter(df_complet['population_totale'], df_complet['nb_voit_elec'], 
                    s=df_complet['nb_bornes']*2,  # taille proportionnelle aux bornes
                    c=df_complet['bornes_pour_1000_hab'], 
                    cmap='RdYlGn', alpha=0.7, edgecolors='black', linewidth=0.5)

# Ligne de tendance
if len(df_complet) > 1:
    z = np.polyfit(df_complet['population_totale'], df_complet['nb_voit_elec'], 1)
    p = np.poly1d(z)
    ax.plot(df_complet['population_totale'], p(df_complet['population_totale']), 
            "r--", alpha=0.8, linewidth=2, label='Tendance')

# Personnalisation
ax.set_xlabel('Population Totale')
ax.set_ylabel('Nombre de Véhicules Électriques')
ax.set_title('Corrélation Population vs VE vs Bornes\n(Taille = nb bornes, Couleur = densité bornes/1000 hab)', 
             fontsize=14, pad=20)

# Colorbar
cbar = plt.colorbar(scatter)
cbar.set_label('Bornes pour 1000 habitants', rotation=270, labelpad=20)

# Ajouter quelques labels pour les points extrêmes
for i, row in df_complet.iterrows():
    if (row['nb_voit_elec'] > df_complet['nb_voit_elec'].quantile(0.9) or 
        row['bornes_pour_1000_hab'] > df_complet['bornes_pour_1000_hab'].quantile(0.9)):
        ax.annotate(row['departement'], 
                   (row['population_totale'], row['nb_voit_elec']),
                   xytext=(5, 5), textcoords='offset points', 
                   fontsize=8, alpha=0.7)

ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

RÉSUMÉ DES INSIGHTS 📊

In [ ]:
# RÉSUMÉ DES INSIGHTS 📊
print("\n" + "="*50)
print("📈 INSIGHTS CLÉS")
print("="*50)

# Calculs des corrélations
corr_pop_ve = df_complet['population_totale'].corr(df_complet['nb_voit_elec'])
corr_ve_bornes = df_complet['nb_voit_elec'].corr(df_complet['nb_bornes'])

print(f"🔗 Corrélation Population ↔ VE : {corr_pop_ve:.3f}")
print(f"🔗 Corrélation VE ↔ Bornes : {corr_ve_bornes:.3f}")

if len(evolution_ve_annuelle) >= 2:
    croissance_ve = ((evolution_ve_annuelle['nb_voit_elec'].iloc[-1] - evolution_ve_annuelle['nb_voit_elec'].iloc[0]) 
                     / evolution_ve_annuelle['nb_voit_elec'].iloc[0] * 100)
    print(f"📈 Croissance VE totale : +{croissance_ve:.1f}% ({evolution_ve_annuelle['annee'].iloc[0]}-{evolution_ve_annuelle['annee'].iloc[-1]})")

print(f"🎯 Départements avec ratio optimal (≥0.1) : {(df_complet['ratio_bornes_ve'] >= 0.1).sum()}/{len(df_complet)}")
print("✅ Visualisations terminées !")


VISUALISATIONS ESSENTIELLES 📈

In [ ]:
# VISUALISATIONS ESSENTIELLES 📈

print("\n" + "="*50)
print("📊 CRÉATION DES 4 VISUALISATIONS ESSENTIELLES")
print("="*50)

# Configuration matplotlib/seaborn
plt.style.use('default')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 10

1. ÉVOLUTION DU PARC DE VÉHICULES ÉLECTRIQUES (dernier recensement/année)

In [ ]:
# ============================================================================
# 1. ÉVOLUTION DU PARC DE VÉHICULES ÉLECTRIQUES (dernier recensement/année)
# ============================================================================

print("📈 Préparation évolution VE (dernier recensement par année)...")

# Nettoyer et préparer les données VE
vehicules_clean = vehicules_df.dropna(subset=['date_recensement', 'nb_voit_elec']).copy()
vehicules_clean['annee'] = vehicules_clean['date_recensement'].dt.year

# IMPORTANT: Prendre le DERNIER recensement de chaque année (pas de cumul)
evolution_ve_annuelle = (vehicules_clean
                        .sort_values('date_recensement')
                        .groupby('annee')
                        .agg({
                            'nb_voit_elec': 'sum',  # Somme par année (tous départements)
                            'date_recensement': 'last'  # Dernière date de l'année
                        })
                        .reset_index())

print(f"✅ Données VE préparées : {len(evolution_ve_annuelle)} années")

2. ÉVOLUTION DU DÉPLOIEMENT DE BORNES DE RECHARGE (cumulé)

In [ ]:
# ============================================================================
# 2. ÉVOLUTION DU DÉPLOIEMENT DE BORNES DE RECHARGE (cumulé)
# ============================================================================

print("🔌 Préparation évolution bornes (déploiement cumulé)...")

# Nettoyer les données bornes
bornes_clean = gdf_merged.dropna(subset=['date_mise_en_service']).copy()
bornes_clean['annee_mois'] = bornes_clean['date_mise_en_service'].dt.to_period('M')

# Compter les nouvelles bornes par mois
nouvelles_bornes_mois = (bornes_clean
                        .groupby('annee_mois')
                        .size()
                        .reset_index(name='nouvelles_bornes'))

# Calculer le cumul (ici c'est légitime car c'est du déploiement d'infrastructure)
nouvelles_bornes_mois['bornes_cumulees'] = nouvelles_bornes_mois['nouvelles_bornes'].cumsum()
nouvelles_bornes_mois['date'] = nouvelles_bornes_mois['annee_mois'].dt.to_timestamp()

print(f"✅ Données bornes préparées : {len(nouvelles_bornes_mois)} mois")

3. PRÉPARER DONNÉES POUR CORRÉLATIONS

In [ ]:
# ============================================================================
# 3. PRÉPARER DONNÉES POUR CORRÉLATIONS
# ============================================================================

print("🔗 Préparation données corrélations...")

# Données par département pour corrélations
ve_par_dep = vehicules_clean.groupby('departement')['nb_voit_elec'].sum().reset_index()
ve_par_dep['departement'] = ve_par_dep['departement'].astype(str).str.strip().str.upper()

# Joindre toutes les données
df_correlations = df_dep.merge(ve_par_dep, on='departement', how='inner')
print(f"✅ Données corrélations : {len(df_correlations)} départements")

CRÉATION DES 4 GRAPHIQUES

In [ ]:
# ============================================================================
# CRÉATION DES 4 GRAPHIQUES
# ============================================================================

fig = plt.figure(figsize=(16, 12))

GRAPHIQUE 1 : ÉVOLUTION PARC VE (TOP LEFT)

In [ ]:
# ============================================================================
# GRAPHIQUE 1 : ÉVOLUTION PARC VE (TOP LEFT)
# ============================================================================
ax1 = plt.subplot(2, 2, 1)

if len(evolution_ve_annuelle) > 1:
    ax1.plot(evolution_ve_annuelle['annee'], evolution_ve_annuelle['nb_voit_elec'], 
             marker='o', linewidth=3, markersize=8, color='blue', alpha=0.8)
    
    ax1.set_xlabel('Année')
    ax1.set_ylabel('Nombre de Véhicules Électriques')
    ax1.set_title('Évolution du Parc de Véhicules Électriques\n(dernier recensement par année)', fontweight='bold')
    ax1.grid(True, alpha=0.3)
    
    # Format des nombres sur l'axe Y
    ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{int(x):,}'))
    
    # Ajouter valeurs sur les points
    for i, row in evolution_ve_annuelle.iterrows():
        ax1.annotate(f'{int(row["nb_voit_elec"]):,}', 
                    (row['annee'], row['nb_voit_elec']),
                    textcoords="offset points", xytext=(0,15), 
                    ha='center', fontsize=9, fontweight='bold')
    
    # Calculer et afficher la croissance
    if len(evolution_ve_annuelle) > 1:
        croissance = ((evolution_ve_annuelle['nb_voit_elec'].iloc[-1] - 
                      evolution_ve_annuelle['nb_voit_elec'].iloc[0]) / 
                      evolution_ve_annuelle['nb_voit_elec'].iloc[0] * 100)
        ax1.text(0.05, 0.95, f'Croissance totale: +{croissance:.1f}%', 
                transform=ax1.transAxes, fontsize=10, 
                bbox=dict(boxstyle="round,pad=0.3", facecolor="lightblue", alpha=0.7),
                verticalalignment='top')

GRAPHIQUE 2 : ÉVOLUTION DÉPLOIEMENT BORNES (TOP RIGHT)

In [ ]:
# ============================================================================
# GRAPHIQUE 2 : ÉVOLUTION DÉPLOIEMENT BORNES (TOP RIGHT)
# ============================================================================
ax2 = plt.subplot(2, 2, 2)

if len(nouvelles_bornes_mois) > 1:
    # Graphique en aires empilées
    ax2.fill_between(nouvelles_bornes_mois['date'], nouvelles_bornes_mois['bornes_cumulees'], 
                    alpha=0.3, color='green', label='Bornes cumulées')
    ax2.plot(nouvelles_bornes_mois['date'], nouvelles_bornes_mois['bornes_cumulees'], 
            linewidth=3, color='darkgreen', marker='s', markersize=4)
    
    ax2.set_xlabel('Date')
    ax2.set_ylabel('Nombre de Bornes')
    ax2.set_title('Évolution du Déploiement de Bornes\n(cumul des installations)', fontweight='bold')
    ax2.grid(True, alpha=0.3)
    
    # Format des dates
    ax2.tick_params(axis='x', rotation=45)
    
    # Afficher quelques valeurs clés
    step = max(1, len(nouvelles_bornes_mois)//5)
    for i in range(0, len(nouvelles_bornes_mois), step):
        row = nouvelles_bornes_mois.iloc[i]
        ax2.annotate(f'{int(row["bornes_cumulees"]):,}', 
                    (row['date'], row['bornes_cumulees']),
                    textcoords="offset points", xytext=(0,10), 
                    ha='center', fontsize=8)
    
    # Stats récentes
    if len(nouvelles_bornes_mois) >= 12:
        nouvelles_12_mois = nouvelles_bornes_mois['nouvelles_bornes'].tail(12).sum()
        ax2.text(0.05, 0.95, f'Nouvelles bornes (12 derniers mois): {nouvelles_12_mois:,}', 
                transform=ax2.transAxes, fontsize=10,
                bbox=dict(boxstyle="round,pad=0.3", facecolor="lightgreen", alpha=0.7),
                verticalalignment='top')


GRAPHIQUE 3 : CORRÉLATION POPULATION VS BORNES (BOTTOM LEFT)

In [ ]:
# ============================================================================
# GRAPHIQUE 3 : CORRÉLATION POPULATION VS BORNES (BOTTOM LEFT)
# ============================================================================
ax3 = plt.subplot(2, 2, 3)

x_pop = df_correlations['population_totale']
y_bornes = df_correlations['nb_bornes']

# Scatter plot
ax3.scatter(x_pop, y_bornes, alpha=0.7, s=80, color='purple', edgecolors='black', linewidth=0.5)

# Ligne de tendance
z = np.polyfit(x_pop, y_bornes, 1)
p = np.poly1d(z)
ax3.plot(x_pop, p(x_pop), "r--", alpha=0.8, linewidth=2, label='Tendance')

# Calcul corrélation
correlation_pop = x_pop.corr(y_bornes)

ax3.set_xlabel('Population Totale')
ax3.set_ylabel('Nombre de Bornes')
ax3.set_title('Corrélation Population vs Bornes de Recharge', fontweight='bold')
ax3.grid(True, alpha=0.3)
ax3.legend()

# Afficher la corrélation
ax3.text(0.05, 0.95, f'Corrélation: {correlation_pop:.3f}', 
        transform=ax3.transAxes, fontsize=12, fontweight='bold',
        bbox=dict(boxstyle="round,pad=0.3", facecolor="lightyellow", alpha=0.8),
        verticalalignment='top')

# Format des axes
ax3.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{int(x):,}'))
ax3.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{int(x):,}'))

GRAPHIQUE 4 : CORRÉLATION VE VS BORNES (BOTTOM RIGHT)

In [ ]:
# ============================================================================
# GRAPHIQUE 4 : CORRÉLATION VE VS BORNES (BOTTOM RIGHT)
# ============================================================================
ax4 = plt.subplot(2, 2, 4)

x_ve = df_correlations['nb_voit_elec']
y_bornes_ve = df_correlations['nb_bornes']

# Scatter plot
ax4.scatter(x_ve, y_bornes_ve, alpha=0.7, s=80, color='orange', edgecolors='black', linewidth=0.5)

# Ligne de tendance
z2 = np.polyfit(x_ve, y_bornes_ve, 1)
p2 = np.poly1d(z2)
ax4.plot(x_ve, p2(x_ve), "r--", alpha=0.8, linewidth=2, label='Tendance')

# Ligne objectif AFI (1 borne pour 10 VE)
max_ve = x_ve.max()
ax4.plot([0, max_ve], [0, max_ve/10], "g--", alpha=0.6, linewidth=2, label='Objectif AFI (1:10)')

# Calcul corrélation
correlation_ve = x_ve.corr(y_bornes_ve)

ax4.set_xlabel('Nombre de Véhicules Électriques')
ax4.set_ylabel('Nombre de Bornes')
ax4.set_title('Corrélation VE vs Bornes de Recharge', fontweight='bold')
ax4.grid(True, alpha=0.3)
ax4.legend()

# Afficher la corrélation
ax4.text(0.05, 0.95, f'Corrélation: {correlation_ve:.3f}', 
        transform=ax4.transAxes, fontsize=12, fontweight='bold',
        bbox=dict(boxstyle="round,pad=0.3", facecolor="lightcoral", alpha=0.8),
        verticalalignment='top')

# Format des axes
ax4.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{int(x):,}'))
ax4.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{int(x):,}'))

RÉSUMÉ STATISTIQUES

In [ ]:
# ============================================================================
# RÉSUMÉ STATISTIQUES
# ============================================================================

print("\n" + "="*50)
print("📊 RÉSUMÉ DES ANALYSES")
print("="*50)

if len(evolution_ve_annuelle) > 1:
    croissance_ve = ((evolution_ve_annuelle['nb_voit_elec'].iloc[-1] - 
                     evolution_ve_annuelle['nb_voit_elec'].iloc[0]) / 
                     evolution_ve_annuelle['nb_voit_elec'].iloc[0] * 100)
    print(f"🚗 Croissance VE totale : +{croissance_ve:.1f}%")
    print(f"📅 Période analysée VE : {evolution_ve_annuelle['annee'].min()}-{evolution_ve_annuelle['annee'].max()}")

if len(nouvelles_bornes_mois) > 1:
    total_bornes = nouvelles_bornes_mois['bornes_cumulees'].iloc[-1]
    print(f"🔌 Total bornes déployées : {total_bornes:,}")
    if len(nouvelles_bornes_mois) >= 12:
        nouvelles_12_mois = nouvelles_bornes_mois['nouvelles_bornes'].tail(12).sum()
        print(f"📈 Nouvelles bornes (12 derniers mois) : {nouvelles_12_mois:,}")

print(f"🔗 Corrélation Population/Bornes : {correlation_pop:.3f}")
print(f"⚡ Corrélation VE/Bornes : {correlation_ve:.3f}")
print(f"📊 Départements analysés : {len(df_correlations)}")

print("\n✅ Les 4 visualisations essentielles ont été générées !")


GRAPHIQUES AGREGES

In [ ]:
# ============================================================================
# CRÉATION DES 4 GRAPHIQUES
# ============================================================================

fig = plt.figure(figsize=(16, 12))

# ============================================================================
# GRAPHIQUE 1 : ÉVOLUTION PARC VE (TOP LEFT)
# ============================================================================
ax1 = plt.subplot(2, 2, 1)

if len(evolution_ve_annuelle) > 1:
    ax1.plot(evolution_ve_annuelle['annee'], evolution_ve_annuelle['nb_voit_elec'], 
             marker='o', linewidth=3, markersize=8, color='blue', alpha=0.8)
    
    ax1.set_xlabel('Année')
    ax1.set_ylabel('Nombre de Véhicules Électriques')
    ax1.set_title('Évolution du Parc de Véhicules Électriques\n(dernier recensement par année)', fontweight='bold')
    ax1.grid(True, alpha=0.3)
    
    # Format des nombres sur l'axe Y
    ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{int(x):,}'))
    
    # Ajouter valeurs sur les points
    for i, row in evolution_ve_annuelle.iterrows():
        ax1.annotate(f'{int(row["nb_voit_elec"]):,}', 
                    (row['annee'], row['nb_voit_elec']),
                    textcoords="offset points", xytext=(0,15), 
                    ha='center', fontsize=9, fontweight='bold')
    
    # Calculer et afficher la croissance
    if len(evolution_ve_annuelle) > 1:
        croissance = ((evolution_ve_annuelle['nb_voit_elec'].iloc[-1] - 
                      evolution_ve_annuelle['nb_voit_elec'].iloc[0]) / 
                      evolution_ve_annuelle['nb_voit_elec'].iloc[0] * 100)
        ax1.text(0.05, 0.95, f'Croissance totale: +{croissance:.1f}%', 
                transform=ax1.transAxes, fontsize=10, 
                bbox=dict(boxstyle="round,pad=0.3", facecolor="lightblue", alpha=0.7),
                verticalalignment='top')


# ============================================================================
# GRAPHIQUE 2 : ÉVOLUTION DÉPLOIEMENT BORNES (TOP RIGHT)
# ============================================================================
ax2 = plt.subplot(2, 2, 2)

if len(nouvelles_bornes_mois) > 1:
    # Graphique en aires empilées
    ax2.fill_between(nouvelles_bornes_mois['date'], nouvelles_bornes_mois['bornes_cumulees'], 
                    alpha=0.3, color='green', label='Bornes cumulées')
    ax2.plot(nouvelles_bornes_mois['date'], nouvelles_bornes_mois['bornes_cumulees'], 
            linewidth=3, color='darkgreen', marker='s', markersize=4)
    
    ax2.set_xlabel('Date')
    ax2.set_ylabel('Nombre de Bornes')
    ax2.set_title('Évolution du Déploiement de Bornes\n(cumul des installations)', fontweight='bold')
    ax2.grid(True, alpha=0.3)
    
    # Format des dates
    ax2.tick_params(axis='x', rotation=45)
    
    # Afficher quelques valeurs clés
    step = max(1, len(nouvelles_bornes_mois)//5)
    for i in range(0, len(nouvelles_bornes_mois), step):
        row = nouvelles_bornes_mois.iloc[i]
        ax2.annotate(f'{int(row["bornes_cumulees"]):,}', 
                    (row['date'], row['bornes_cumulees']),
                    textcoords="offset points", xytext=(0,10), 
                    ha='center', fontsize=8)
    
    # Stats récentes
    if len(nouvelles_bornes_mois) >= 12:
        nouvelles_12_mois = nouvelles_bornes_mois['nouvelles_bornes'].tail(12).sum()
        ax2.text(0.05, 0.95, f'Nouvelles bornes (12 derniers mois): {nouvelles_12_mois:,}', 
                transform=ax2.transAxes, fontsize=10,
                bbox=dict(boxstyle="round,pad=0.3", facecolor="lightgreen", alpha=0.7),
                verticalalignment='top')

# ============================================================================
# GRAPHIQUE 3 : CORRÉLATION POPULATION VS BORNES (BOTTOM LEFT)
# ============================================================================
ax3 = plt.subplot(2, 2, 3)

x_pop = df_correlations['population_totale']
y_bornes = df_correlations['nb_bornes']

# Scatter plot
ax3.scatter(x_pop, y_bornes, alpha=0.7, s=80, color='purple', edgecolors='black', linewidth=0.5)

# Ligne de tendance
z = np.polyfit(x_pop, y_bornes, 1)
p = np.poly1d(z)
ax3.plot(x_pop, p(x_pop), "r--", alpha=0.8, linewidth=2, label='Tendance')

# Calcul corrélation
correlation_pop = x_pop.corr(y_bornes)

ax3.set_xlabel('Population Totale')
ax3.set_ylabel('Nombre de Bornes')
ax3.set_title('Corrélation Population vs Bornes de Recharge', fontweight='bold')
ax3.grid(True, alpha=0.3)
ax3.legend()

# Afficher la corrélation
ax3.text(0.05, 0.95, f'Corrélation: {correlation_pop:.3f}', 
        transform=ax3.transAxes, fontsize=12, fontweight='bold',
        bbox=dict(boxstyle="round,pad=0.3", facecolor="lightyellow", alpha=0.8),
        verticalalignment='top')

# Format des axes
ax3.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{int(x):,}'))
ax3.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{int(x):,}'))

# ============================================================================
# GRAPHIQUE 4 : CORRÉLATION VE VS BORNES (BOTTOM RIGHT)
# ============================================================================
ax4 = plt.subplot(2, 2, 4)

x_ve = df_correlations['nb_voit_elec']
y_bornes_ve = df_correlations['nb_bornes']

# Scatter plot
ax4.scatter(x_ve, y_bornes_ve, alpha=0.7, s=80, color='orange', edgecolors='black', linewidth=0.5)

# Ligne de tendance
z2 = np.polyfit(x_ve, y_bornes_ve, 1)
p2 = np.poly1d(z2)
ax4.plot(x_ve, p2(x_ve), "r--", alpha=0.8, linewidth=2, label='Tendance')

# Ligne objectif AFI (1 borne pour 10 VE)
max_ve = x_ve.max()
ax4.plot([0, max_ve], [0, max_ve/10], "g--", alpha=0.6, linewidth=2, label='Objectif AFI (1:10)')

# Calcul corrélation
correlation_ve = x_ve.corr(y_bornes_ve)

ax4.set_xlabel('Nombre de Véhicules Électriques')
ax4.set_ylabel('Nombre de Bornes')
ax4.set_title('Corrélation VE vs Bornes de Recharge', fontweight='bold')
ax4.grid(True, alpha=0.3)
ax4.legend()

# Afficher la corrélation
ax4.text(0.05, 0.95, f'Corrélation: {correlation_ve:.3f}', 
        transform=ax4.transAxes, fontsize=12, fontweight='bold',
        bbox=dict(boxstyle="round,pad=0.3", facecolor="lightcoral", alpha=0.8),
        verticalalignment='top')

# Format des axes
ax4.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{int(x):,}'))
ax4.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{int(x):,}'))

# ============================================================================
# FINALISATION
# ============================================================================

plt.tight_layout(pad=3.0)
plt.show()